# 03. Verified v251 reference pipeline

이 노트북은 모델 가중치 없이 v251의 공개 가능한 앙상블과 후처리 인터페이스를 보여줍니다. 예시 배열은 API 확인용이며 대회 예측값이 아닙니다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.reference_pipeline import load_reference_config, run_v251_reference

config = load_reference_config(PROJECT_ROOT / 'configs' / 'v251_reference.json')
config['reference_artifact']

## Score boundary

v251의 기록 점수 `1152.3953935303`과 최종 leaderboard score `1159.52968`은 같은 artifact의 결과로 표현하지 않습니다.

In [ ]:
reference = config['reference_artifact']
final_result = config['competition_result']
print('Verified v251 score:', reference['leaderboard_score'])
print('Final competition score:', final_result['final_leaderboard_score'])
print('Same artifact:', final_result['same_artifact_as_reference'])

## Weight-free execution example

실제 사용 시 각 모델의 확률과 배포 시 계수가 이미 반영된 residual probability delta를 전달합니다. `tm_auto5`는 v251 설정에서 gamma가 0이므로 비활성화되어 있습니다.

In [ ]:
n_rows = 4
members = {
    name: np.linspace(0.42, 0.58, n_rows)
    for name in config['ensemble']['normalized_weights']
}
residuals = {
    name: np.zeros(n_rows)
    for name, settings in config['overlays'].items()
    if settings.get('enabled', True)
}
reference_probability = run_v251_reference(members, residuals, config)
reference_probability

## Private artifacts required for exact inference

- CatBoost and LightGBM model files
- MLP and FT-Transformer weights
- historical lookup tables and residual-model outputs
- competition test data

이 파일들은 저장소에 포함하거나 커밋하지 않습니다.